# 🌌 Mundus 0.1 (3B) — Multi-Game Foundation World Model Training
This notebook trains the **Mundus 0.1 3B World Model** (~3.18 Billion Parameters) on Google Colab GPUs (T4 16GB, L4 24GB, or A100 40GB).

### 🎮 The 4 Flagship Game Universes:
1. **🔴 Doom 3D**: First-Person 3D Raycasting with demons, shotgun, and HUD.
2. **🟢 Mundus Platformer**: 2D side-scrolling physics with parabolic jump arcs, mystery blocks, and Goombas.
3. **🗡️ Mundus Dungeon**: Top-down action RPG with torchlight illumination, skeletons, and sword combat.
4. **🏎️ Mundus Grand Prix**: High-speed circuit racing with drift physics, kerbs, and AI rivals.

### ⚡ Memory Optimizations for 3B Parameters:
- **Mixed Precision (AMP FP16/BF16)**: 2x speedup, 50% activation memory.
- **8-Bit AdamW (bitsandbytes)**: Reduces optimizer memory from 25.4 GB down to 6.36 GB.
- **Gradient Accumulation**: Simulates effective batch size 8 on single GPUs without OOM.

In [ ]:
# 1. Verify GPU Hardware
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Install Dependencies
!pip install --upgrade pip
!pip install bitsandbytes accelerate huggingface_hub pillow numpy tqdm


In [ ]:
# 3. Clone Repository or Update Existing Workspace
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Ensure clean workspace in /content root
%cd /content
if not os.path.exists("/content/mundus-world-model/.git"):
    !rm -rf /content/mundus-world-model
    !git clone https://github.com/tahaenasraoui-debug/mundus-0.1-85-230M.git /content/mundus-world-model
%cd /content/mundus-world-model
!git pull origin main


### Option A: Unified Multi-Game Foundation Training (`--env all`)
Trains Mundus 3B on **all 4 games simultaneously** by mixing trajectories from Doom 3D, Platformer, Dungeon RPG, and Grand Prix Racer into one multi-universe dataset!

In [ ]:
# 4A. Multi-Game Foundation Training (Recommended)
!python3 train.py \
    --model 3b \
    --env all \
    --episodes 1000 \
    --episode_length 40 \
    --steps 50000 \
    --batch_size 1 \
    --grad_accum 8 \
    --lr 2e-4 \
    --amp \
    --adam8bit \
    --grad_checkpoint \
    --device cuda

### Option B: Sequential Training (One Game After the Other)
Train on Doom 3D first, then resume on Platformer, then Dungeon, then Racer!

In [ ]:
# 4B. Sequential Curriculum Training
# Step 1: Doom 3D
!python3 train.py --model 3b --env doom --episodes 400 --steps 15000 --batch_size 1 --grad_accum 8 --amp --adam8bit --grad_checkpoint --device cuda

# Step 2: Resume & Train Platformer
!python3 train.py --model 3b --env platformer --resume checkpoints/mundus_0.1_3b_latest.pt --episodes 400 --steps 15000 --batch_size 1 --grad_accum 8 --amp --adam8bit --grad_checkpoint --device cuda

# Step 3: Resume & Train Dungeon RPG
!python3 train.py --model 3b --env dungeon --resume checkpoints/mundus_0.1_3b_latest.pt --episodes 400 --steps 15000 --batch_size 1 --grad_accum 8 --amp --adam8bit --grad_checkpoint --device cuda

# Step 4: Resume & Train Grand Prix Racer
!python3 train.py --model 3b --env racer --resume checkpoints/mundus_0.1_3b_latest.pt --episodes 400 --steps 15000 --batch_size 1 --grad_accum 8 --amp --adam8bit --grad_checkpoint --device cuda

In [ ]:
# 5. Export / Upload Checkpoints to Hugging Face
from huggingface_hub import HfApi
api = HfApi()
repo_id = "your-username/mundus-world-model-3b"
# api.create_repo(repo_id=repo_id, exist_ok=True)
# api.upload_file(path_or_fileobj="checkpoints/mundus_0.1_3b_latest.pt", path_in_repo="mundus_0.1_3b_latest.pt", repo_id=repo_id)
print("Upload step ready!")